### 1. Import libraries and Reproducibility rules

In [20]:
import json
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt


In [21]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")
print("Pytorch version:", torch.__version__)

Pytorch version: 2.8.0+cpu


### 2. Preprocessing

In [22]:
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=Mean, std=STD)
])

train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(50, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.RandomResizedCrop(
        200, scale=(0.9, 1.0), ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

### 3. Data loading

In [23]:
train_dir = "data/train"
evaluation_dir = "data/test"

evaluation_dataset = datasets.ImageFolder(evaluation_dir, transform=evaluation_transform)
evaluation_loader = DataLoader(evaluation_dataset, batch_size=20, shuffle=False, num_workers=0)

loader_generator = torch.Generator().manual_seed(SEED)

def make_train_loader(transform):
    dataset = datasets.ImageFolder(train_dir, transform=transform)
    return DataLoader(dataset, batch_size=20, shuffle=True, num_workers=0, generator=loader_generator)

baseline_train_loader = make_train_loader(evaluation_transform)

print("Classes:", evaluation_dataset.class_to_idx)
print("Training images:", len(baseline_train_loader.dataset))
print("Evaluation images:", len(evaluation_dataset))

Classes: {'curly': 0, 'straight': 1}
Training images: 800
Evaluation images: 201


### 4. Modelling

The model will be made with said pipeline :
1. Conv layer : Scans the image for patterns 
2. ReLU + MaxPool : Shrinks image and keeps strong signals
3. Flatten: Lines the numbers up in one long row
4. Hidden Layer: Combines the patterns
5. Output Layer: the numbers 

In [24]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

model = Model().to(device)
print(model)

Model(
  (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (hidden): Linear(in_features=313632, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)


### Q1 

The answer is ``nn.BCEwithLogitsLoss()``. it applies to sigmoid function then measures the error

### Q2

The parameters for the model.

In [25]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

total_params = sum(p.numel() for p in model.parameters())
print("Total trainable parameters:", total_params)

Total trainable parameters: 20073473


the total trainable params are ``20073473``

#### Training metrics and evaluation functions

In [26]:
def run_one_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss, total_correct, total_count = 0.0, 0, 0

    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            n = labels.size(0)
            total_loss += loss.item() * n
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (predictions == labels).sum().item()
            total_count += n
    return total_loss / total_count, total_correct / total_count


def run_phase(train_loader, epochs, name):
    history = {"train_loss": [], "train_accuracy": [], "evaluation_loss": [], "evaluation_accuracy": []}
    for epoch in range(1, epochs + 1):
        tl, ta = run_one_epoch(train_loader, train=True)
        el, ea = run_one_epoch(evaluation_loader, train=False)
        history["train_loss"].append(tl)
        history["train_accuracy"].append(ta)
        history["evaluation_loss"].append(el)
        history["evaluation_accuracy"].append(ea)
        print(f"[{name}] epoch {epoch:2d} | train loss {tl:.4f} acc {ta:.4f} | eval loss {el:.4f} acc {ea:.4f}")
    with open(f"history_{name}.json", "w") as f:
        json.dump(history, f, indent=2)
    return history




doing baseline training with 10 epochs

In [27]:
history_baseline = run_phase(baseline_train_loader, epochs=10, name="baseline")

[baseline] epoch  1 | train loss 0.6385 acc 0.6450 | eval loss 0.6010 acc 0.6567
[baseline] epoch  2 | train loss 0.5441 acc 0.7175 | eval loss 0.6003 acc 0.6517
[baseline] epoch  3 | train loss 0.4955 acc 0.7550 | eval loss 0.6405 acc 0.6219
[baseline] epoch  4 | train loss 0.4330 acc 0.7913 | eval loss 0.6274 acc 0.6915
[baseline] epoch  5 | train loss 0.4048 acc 0.8087 | eval loss 0.6293 acc 0.6617
[baseline] epoch  6 | train loss 0.3953 acc 0.8075 | eval loss 0.6332 acc 0.7114
[baseline] epoch  7 | train loss 0.3437 acc 0.8363 | eval loss 0.6895 acc 0.6716
[baseline] epoch  8 | train loss 0.2806 acc 0.8850 | eval loss 0.6497 acc 0.7214
[baseline] epoch  9 | train loss 0.2236 acc 0.9100 | eval loss 0.7114 acc 0.7164
[baseline] epoch 10 | train loss 0.1584 acc 0.9387 | eval loss 1.0212 acc 0.6368


### Q3 & Q4

Standard deviation of baseline 

In [28]:
q3 = round(float(np.median(history_baseline["train_accuracy"])), 2)
q4 = round(float(np.std(history_baseline["train_loss"], ddof=0)), 3)
print("Q3 :", q3)
print("Q4 :", q4)

Q3 : 0.81
Q4 : 0.139


### Training with augmentation

In [29]:
augmented_train_loader = make_train_loader(train_transform_augmented)
history_augmented = run_phase(augmented_train_loader, epochs=10, name="augmented")

[augmented] epoch  1 | train loss 0.6444 acc 0.6400 | eval loss 0.6283 acc 0.6866
[augmented] epoch  2 | train loss 0.5865 acc 0.6787 | eval loss 0.7047 acc 0.6766
[augmented] epoch  3 | train loss 0.5940 acc 0.6613 | eval loss 0.5812 acc 0.7114
[augmented] epoch  4 | train loss 0.5469 acc 0.7150 | eval loss 0.6194 acc 0.6816
[augmented] epoch  5 | train loss 0.5469 acc 0.7113 | eval loss 0.6176 acc 0.7065
[augmented] epoch  6 | train loss 0.5207 acc 0.7275 | eval loss 0.5709 acc 0.7065
[augmented] epoch  7 | train loss 0.5528 acc 0.7037 | eval loss 0.5715 acc 0.7114
[augmented] epoch  8 | train loss 0.5313 acc 0.7250 | eval loss 0.6018 acc 0.6965
[augmented] epoch  9 | train loss 0.5075 acc 0.7450 | eval loss 0.6582 acc 0.6517
[augmented] epoch 10 | train loss 0.4897 acc 0.7638 | eval loss 0.5377 acc 0.7313


### Q5 & Q6 

Mean of augmented ``evaluation_loss`` and mean of the last 5 augmented ``evaluation_accuracy`` values

In [30]:
q5 = round(float(np.mean(history_augmented["evaluation_loss"])), 3)
q6 = round(float(np.mean(history_augmented["evaluation_accuracy"][-5:])), 2)
print("Q5:", q5)
print("Q6:", q6)

Q5: 0.609
Q6: 0.7
